In [3]:
import sqlite3
import random
from datetime import date, timedelta

# ── Configuration ────────────────────────────────────────────
DB_PATH = r"C:\Users\ABHISHEK PANCHAL\OneDrive\Desktop\SQL Project\energy_project.db"
START_DATE = date(2024, 1, 1)
NUM_DAYS = 365
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

# ── Charge windows (off-peak hours) ──────────────────────────
CHARGE_WINDOWS = [
    ("01:00", "05:00"),
    ("02:00", "06:00"),
    ("03:00", "07:00"),
]

# ── Discharge windows (peak hours) ───────────────────────────
DISCHARGE_WINDOWS = [
    ("08:00", "11:00"),
    ("09:00", "12:00"),
    ("16:00", "19:00"),
    ("17:00", "20:00"),
]

# ── Site configuration ────────────────────────────────────────
SITES = [
    {
        "site_id": 1,
        "site_name": "BESS_North",
        "capacity_kwh": 200.0,
        "base_efficiency": 0.93,
        "degradation_rate": 0.00005,
        "energy_in_range": (90, 100),
    },
    {
        "site_id": 2,
        "site_name": "BESS_South",
        "capacity_kwh": 150.0,
        "base_efficiency": 0.91,
        "degradation_rate": 0.00008,
        "energy_in_range": (68, 78),
    },
]

# ── C-rate efficiency penalty (based on Preger et al. 2020) ──
def c_rate_penalty(c_rate):
    """
    Simplified linear RTE penalty based on C-rate.
    At 0.3C: no penalty
    At 0.5C: ~2% penalty
    At 0.7C: ~4% penalty
    At 1.0C: ~7% penalty
    Grounded in Preger et al. (Sandia, J Electrochem Soc 2020)
    """
    penalty = max(0, (c_rate - 0.3) * 0.175)
    return round(penalty, 4)

# ── Connect to database ───────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# ── Clear existing cycle data ─────────────────────────────────
cursor.execute("DELETE FROM battery_cycles")
print("Cleared existing battery_cycles data.")

# ── Generate cycles ───────────────────────────────────────────
cycle_id = 1

for day_num in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_num)

    for site in SITES:
        # Pick random charge and discharge windows
        charge = random.choice(CHARGE_WINDOWS)
        discharge = random.choice(DISCHARGE_WINDOWS)

        # Energy in varies slightly each day
        energy_in = round(random.uniform(*site["energy_in_range"]), 1)

        # Calculate C-rate: energy_in / capacity
        c_rate = round(energy_in / site["capacity_kwh"], 4)

        # Base efficiency degrades over time
        base_eff = (
            site["base_efficiency"]
            - (day_num * site["degradation_rate"])
            + random.uniform(-0.005, 0.005)
        )

        # Apply C-rate penalty on top of degradation
        penalty = c_rate_penalty(c_rate)
        efficiency = base_eff - penalty
        efficiency = max(0.80, min(0.98, efficiency))

        energy_out = round(energy_in * efficiency, 1)

        # SoC varies realistically
        soc_start = round(random.uniform(8, 20), 1)
        soc_end = round(random.uniform(82, 95), 1)

        cursor.execute("""
            INSERT INTO battery_cycles VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            cycle_id,
            site["site_id"],
            current_date.isoformat(),
            charge[0],
            charge[1],
            discharge[0],
            discharge[1],
            energy_in,
            energy_out,
            soc_start,
            soc_end,
            c_rate,
        ))

        cycle_id += 1

conn.commit()
conn.close()

print(f"Done — {cycle_id - 1} cycles inserted with C-rate data.")

Cleared existing battery_cycles data.
Done — 730 cycles inserted with C-rate data.
